In [ ]:
import sympy as sp

# ---------------------------------------------------------
# Time and time-dependent generalized coordinates
# ---------------------------------------------------------
t = sp.symbols('t')
phi1 = sp.Function('phi1')(t)
phi2 = sp.Function('phi2')(t)

w1 = sp.diff(phi1, t)
w2 = sp.diff(phi2, t)
a1 = sp.diff(phi1, t, 2)
a2 = sp.diff(phi2, t, 2)

# ---------------------------------------------------------
# Constant parameters
# ---------------------------------------------------------
m1, m2 = sp.symbols('m1 m2', positive=True)
J1, J2 = sp.symbols('J1 J2', positive=True)
r1, l1, r2 = sp.symbols('r1 l1 r2', positive=True)   # r1: to m1 COM, l1: full link-1 length to joint2, r2: to m2 COM
B1, B2 = sp.symbols('B1 B2', nonnegative=True)        # bearing damping coeffs
g = sp.symbols('g', positive=True)
Fm = sp.symbols('Fm')                                  # motor torque on joint 1

# ---------------------------------------------------------
# Positions (ground frame, origin at joint 1, y up)
# ---------------------------------------------------------
x1 = r1 * sp.sin(phi1)
y1 = -r1 * sp.cos(phi1)

xa = l1 * sp.sin(phi1)          # point where link1 meets link2
ya = -l1 * sp.cos(phi1)

x2 = xa + r2 * sp.sin(phi1 + phi2)
y2 = ya - r2 * sp.cos(phi1 + phi2)


# ---------------------------------------------------------
# Velocities (sympy does the chain rule automatically)
# ---------------------------------------------------------
v1x, v1y = sp.diff(x1, t), sp.diff(y1, t)
v2x, v2y = sp.diff(x2, t), sp.diff(y2, t)

v1_sq = sp.trigsimp(sp.expand(v1x**2 + v1y**2))
v2_sq = sp.trigsimp(sp.expand(v2x**2 + v2y**2))


sp.pprint(sp.simplify(v2x))
sp.pprint(sp.simplify(v2y))

print("=== v1^2 ===")
sp.pprint(v1_sq)
print("\n=== v2^2 ===")
sp.pprint(sp.simplify(v2_sq))

# ---------------------------------------------------------
# Kinetic / potential energy
# ---------------------------------------------------------
Ke = sp.Rational(1,2)*J1*w1**2 + sp.Rational(1,2)*J2*(w1+w2)**2 \
     + sp.Rational(1,2)*m1*v1_sq + sp.Rational(1,2)*m2*v2_sq

Pe = m1*g*y1 + m2*g*y2

L = sp.simplify(Ke - Pe)

# ---------------------------------------------------------
# Rayleigh dissipation function (bearings)
# ---------------------------------------------------------
Fb = sp.Rational(1,2)*B1*w1**2 + sp.Rational(1,2)*B2*w2**2

# ---------------------------------------------------------
# Generalized forces (motor torque + damping via dissipation fn)
# ---------------------------------------------------------
Q1 = Fm - sp.diff(Fb, w1)
Q2 = -sp.diff(Fb, w2)

# ---------------------------------------------------------
# Euler-Lagrange equations:  d/dt(dL/dw_i) - dL/dphi_i - Q_i = 0
# ---------------------------------------------------------
EL1 = sp.diff(sp.diff(L, w1), t) - sp.diff(L, phi1) - Q1
EL2 = sp.diff(sp.diff(L, w2), t) - sp.diff(L, phi2) - Q2

EL1 = sp.simplify(EL1)
EL2 = sp.simplify(EL2)

print("\n=== Equation 1 (== 0) ===")
sp.pprint(EL1)
print("\n=== Equation 2 (== 0) ===")
sp.pprint(EL2)

# ---------------------------------------------------------
# Extract clean mass matrix M, and RHS vector, by linearity in a1,a2
# EL_i = M_i1*a1 + M_i2*a2 + rest_i   =>   rest_i = EL_i with a1=a2=0
# ---------------------------------------------------------
M11 = sp.diff(EL1, a1)
M12 = sp.diff(EL1, a2)
M21 = sp.diff(EL2, a1)
M22 = sp.diff(EL2, a2)

M = sp.Matrix([[M11, M12], [M21, M22]])

rest1 = sp.simplify(EL1 - (M11*a1 + M12*a2))   # = -Q1 + C1 + G1  (everything not multiplying an accel)
rest2 = sp.simplify(EL2 - (M21*a1 + M22*a2))

RHS = -sp.Matrix([rest1, rest2])  # so that M*[a1,a2]^T = RHS

print("\n=== Mass matrix M ===")
sp.pprint(sp.simplify(M))
print("\n=== M symmetric check (M12 - M21, should be 0) ===")
sp.pprint(sp.simplify(M12 - M21))
print("\n=== RHS (Fm, damping, Coriolis, gravity all folded in) ===")
sp.pprint(sp.simplify(RHS))

# ---------------------------------------------------------
# Solve via matrix inverse
# ---------------------------------------------------------
accel = sp.simplify(M.inv() * RHS)
phi1_ddot = accel[0]
phi2_ddot = accel[1]


print("\n=== phi1_ddot ===")
sp.pprint(sp.simplify(phi1_ddot), num_columns=200)

print("\n=== phi2_ddot ===")
sp.pprint(sp.simplify(phi2_ddot), num_columns=200)






In [ ]:
#write to matlab func


# 1. swap time-dependent Functions/Derivatives for plain symbols -- octave_code
#    can only print ordinary symbols, not phi1(t) or Derivative(phi1(t), t)
phi1_s, phi2_s, w1_s, w2_s = sp.symbols('phi1 phi2 w1 w2')
accel_plain = accel.subs({w1: w1_s, w2: w2_s, phi1: phi1_s, phi2: phi2_s})

# 2. common subexpression elimination -- without this, M.inv() produces one
#    gigantic line per output since the determinant/products repeat many times
replacements, reduced = sp.cse([accel_plain[0], accel_plain[1]], optimizations='basic')

# 3. assemble as a MATLAB/Octave function
lines = ["function [a1, a2] = double_pendulum_accel(phi1, phi2, w1, w2, m1, m2, J1, J2, r1, l1, r2, B1, B2, g, Fm)"]
for sym, expr in replacements:
    lines.append(f"{sym} = {sp.octave_code(expr)};")
lines.append(f"a1 = {sp.octave_code(reduced[0])};")
lines.append(f"a2 = {sp.octave_code(reduced[1])};")
lines.append("end")

with open('double_pendulum_accel.m', 'w') as f:
    f.write("\n".join(lines))

In [ ]:
# ---------------------------------------------------------
# Lambdify for numeric integration
# state = [phi1, phi2, w1, w2] -> returns [a1, a2]
# ---------------------------------------------------------
params = (m1, m2, J1, J2, r1, l1, r2, B1, B2, g, Fm)
accel_func = sp.lambdify((phi1, phi2, w1, w2) + params, accel, 'numpy')

print("\naccel_func(phi1, phi2, w1, w2, m1,m2,J1,J2,r1,l1,r2,B1,B2,g,Fm) is ready for use in an ODE solver (e.g. scipy.integrate.solve_ivp).")



# ---------------------------------------------------------
# Numeric integration with scipy.integrate.solve_ivp
# ---------------------------------------------------------
from scipy.integrate import solve_ivp
import numpy as np

check = {Fm: 10, B1: 0.2, B2: 0.3, m1: 1, m2: 1, J1: 0.1, J2: 0.1,
          r1: 0.9, l1: 1, r2: 0.8, g: 9.81}

# pull numeric values in the SAME order as `params`
param_values = [check[p] for p in params]

def deriv(t, y):
    phi1_, phi2_, w1_, w2_ = y
    a1_, a2_ = accel_func(phi1_, phi2_, w1_, w2_, *param_values).flatten()
    return [w1_, w2_, a1_, a2_]

y0 = [0.0, 0.0, 0.0, 0.0]   # initial phi1, phi2, w1, w2
t_span = (0, 10)
t_eval = np.linspace(*t_span, 1000)

sol = solve_ivp(deriv, t_span, y0, t_eval=t_eval, method='RK45', rtol=1e-8, atol=1e-8)

print("solve_ivp success:", sol.success, sol.message)
print("final state:", sol.y[:, -1])




In [ ]:

# ---------------------------------------------------------
# Animation: render swinging double pendulum to mp4
# ---------------------------------------------------------
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.animation as animation

# free swing: no motor torque, displaced start, let gravity + bearing damping act
check_anim = dict(check)
param_values_anim = [check_anim[p] for p in params]

def deriv_anim(t, y):
    phi1_, phi2_, w1_, w2_ = y
    a1_, a2_ = accel_func(phi1_, phi2_, w1_, w2_, *param_values_anim).flatten()
    return [w1_, w2_, a1_, a2_]

y0_anim = y0# displaced start, released from rest
t_span_anim = (0, 10)
fps = 30
t_eval_anim = np.linspace(*t_span_anim, int((t_span_anim[1]-t_span_anim[0])*fps))

sol_anim = solve_ivp(deriv_anim, t_span_anim, y0_anim, t_eval=t_eval_anim,
                      method='RK45', rtol=1e-9, atol=1e-9)

phi1_t = sol_anim.y[0]
phi2_t = sol_anim.y[1]

l1_val = check_anim[l1]
r2_val = check_anim[r2]

# joint positions over time (x right, y up; origin at fixed pivot)
x1 = l1_val * np.sin(phi1_t)
y1 = -l1_val * np.cos(phi1_t)
x2 = x1 + r2_val * np.sin(phi1_t + phi2_t)
y2 = y1 - r2_val * np.cos(phi1_t + phi2_t)

reach = l1_val + r2_val
fig, ax = plt.subplots(figsize=(5, 5))
ax.set_xlim(-reach*1.2, reach*1.2)
ax.set_ylim(-reach*1.2, reach*1.2)
ax.set_aspect('equal')
ax.grid(True, alpha=0.3)

line, = ax.plot([], [], 'o-', lw=2, color='steelblue', markersize=8)
trace, = ax.plot([], [], '-', lw=1, color='lightcoral', alpha=0.6)
time_text = ax.text(0.05, 0.95, '', transform=ax.transAxes)

trace_x, trace_y = [], []

def init():
    line.set_data([], [])
    trace.set_data([], [])
    time_text.set_text('')
    return line, trace, time_text

def update(frame):
    xs = [0, x1[frame], x2[frame]]
    ys = [0, y1[frame], y2[frame]]
    line.set_data(xs, ys)

    trace_x.append(x2[frame])
    trace_y.append(y2[frame])
    trace.set_data(trace_x, trace_y)

    time_text.set_text(f't = {sol_anim.t[frame]:.2f}s')
    return line, trace, time_text

ani = animation.FuncAnimation(fig, update, frames=len(sol_anim.t),
                                init_func=init, blit=True, interval=1000/fps)

ani.save('/home/aizej/Desktop/disky/500GB_2025/gen/kodezy/python/mujoco_RL/physic_vid/double_pendulum.mp4', writer='ffmpeg', fps=fps, dpi=150)
print("saved video")